In [ ]:
# https://bura.brunel.ac.uk/bitstream/2438/28335/3/FullText.pdf

In [ ]:
import numpy as np
import cv2

## Filter

In [ ]:
def white_balance(img, Wb, Wg, Wr):
    I_copy = img.copy()
    I_copy[:, :, 0] *= Wb
    I_copy[:, :, 1] *= Wg
    I_copy[:, :, 2] *= Wr
    return np.clip(I_copy, 0, 255).astype(np.uint8)

# γ < 1 is sometimes called an encoding gamma, and the process of encoding with this compressive power-law nonlinearity is called gamma compression.
# γ > 1 is called a decoding gamma, and the application of the expansive power-law nonlinearity is called gamma expansion.
def gamma_correction(img, gamma):
    I_copy = img.copy()
    I_copy = I_copy.astype(np.float32) / 255.0
    I_copy = np.power(I_copy, gamma)
    return (I_copy * 255).clip(0, 255).astype(np.uint8)

def tone_filter(img, t_params):
    # Normalize image to [0,1]
    I_copy = img.astype(np.float32) / 255.0

    # prefix Tk = cumulative sum t0..t(k-1)
    T_prefix = np.cumsum(t_params)
    T8 = max(T_prefix[-1], 1e-8)  # total sum

    # I_tone in range [0,1]
    I_tone = np.zeros_like(I_copy)

    sigma = np.zeros_like(I_tone)
    for k in range(8):
        value = np.clip(8 * I_copy - k, 0, 1) * T_prefix[k]
        sigma += value
    I_tone = sigma / T8
    return (I_tone * 255).clip(0, 255).astype(np.uint8)

def HE_filter(img, alpha):
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    hsv[:,:,2] = cv2.equalizeHist(hsv[:,:,2])
    HE = cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)
    HE_img = alpha * HE.astype(np.float32) + (1 - alpha) * img.astype(np.float32)
    HE_img = np.clip(HE_img, 0, 255).astype(np.uint8)
    return HE_img

def Laplace_filter(img, lambda_var):
    I_copy = img.copy().astype(np.float32)

    # Compute Laplacian for each channels
    lap_op = cv2.Laplacian(I_copy, cv2.CV_32F, ksize=3)

    # Condition masks
    mask_pos = lap_op > 0
    mask_neg = lap_op < 0

    I_L = I_copy.copy()
    I_L[mask_neg] = I_copy[mask_neg] + lambda_var * (I_copy[mask_neg] - lap_op[mask_neg])
    I_L[mask_pos] = I_copy[mask_pos] + lambda_var * (I_copy[mask_pos] + lap_op[mask_pos])
    
    return np.clip(I_L, 0, 255).astype(np.uint8)

## Dehaze filter

In [17]:
def DarkChannel(I, sz=15):
    b, g, r = cv2.split(I)
    dc = cv2.min(cv2.min(r, g), b)
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (sz, sz))
    dark_channel = cv2.erode(dc, kernel)
    return dark_channel

def Ambient_Light_Estimate(I, dark_channel, p=1e-3):
    h, w = dark_channel.shape
    N = h * w
    # Flat and sort dark channel
    flat_dark = dark_channel.reshape(N)
    sorted_dark = flat_dark.argsort()
    sorted_dark = sorted_dark[::-1]
    flat_I = I.reshape(N, 3)       # 3D->2D

    # Number of pixel used for the estimation
    M = max(1, int(p * N))    
    M = min(M, N)

    A = np.zeros([1, 3], dtype=np.float32)
    for i in range(M):
        A += flat_I[sorted_dark[i]]            # Add pixel color
    A /= M
    return A

def Transmission_Map_Estimate(I, A, sz, w=0.95):
    copy_I = np.empty(I.shape, I.dtype)
    for c in range(3):
        copy_I[:, :, c] = I[:, :, c] / A[0, c]
    t_x = 1 - w * DarkChannel(copy_I, sz)
    return t_x

# I is one channel of image and base [0, 1]
# accumulated values of image (2D prefix sum)
def Integral_Image(I):
    return np.cumsum(np.cumsum(I, axis=0), axis=1)

# only apply for one channel
def BoundaryHandling(I, rr):
    return np.pad(I, ((rr, rr), (rr, rr)), mode='reflect')

# I is one channel of img and base [0, 1]
# rr is radius of patch (r = 2rr + 1, patch rxr)
def Patch_Averages(I, rr):
    return cv2.boxFilter(I,cv2.CV_64F,(rr,rr))

def Guided_Filter(I, t, rr, eps):
    mean_I = Patch_Averages(I, rr)
    mean_I2 = Patch_Averages(I*I, rr)
    var_I = mean_I2 - mean_I * mean_I
    mean_t = Patch_Averages(t, rr)
    mean_It = Patch_Averages(I * t, rr)
    covar_It = mean_It - mean_I * mean_t
    a = covar_It / (var_I + eps)
    b = mean_t - a * mean_I
    mean_a = Patch_Averages(a, rr)
    mean_b = Patch_Averages(b, rr)
    t_0 = mean_a * I + mean_b
    return t_0

def TransmissionRefine(I, t, r=61, eps=1e-3):
    I_gray = cv2.cvtColor(I,cv2.COLOR_BGR2GRAY);
    t = Guided_Filter(I_gray, t, r, eps);
    t = np.clip(t, 0, 1)
    return t

def Recover_Scene_Radiance(I, t, A, t0=0.1):
    I_dehaze = np.zeros_like(I, dtype=float)
    for c in range(3):
        I_dehaze[:, :, c] = (I[:, :, c] - A[0, c]) / np.maximum(t, t0) + A[0, c]
    I_dehaze = np.clip(I_dehaze, 0, 1)
    return I_dehaze

# Using Dark Channel Prior (DCP) Algorithm
# I: base [0, 255]
# return base [0, 1]
def Dehazing(I, sz=15, p=1e-3, w=0.95, r=61, eps=0.0001, t0=0.1):
    I = I.astype(np.float32) / 255.0        # Normalize to base [0, 1]
    J_dark = DarkChannel(I, sz)
    A = Ambient_Light_Estimate(I, J_dark, p)
    t = Transmission_Map_Estimate(I, A, sz, w)
    t_refine = TransmissionRefine(I, t, r, eps)
    J = Recover_Scene_Radiance(I, t_refine, A, t0)
    return J